In [1]:
import pandas as pd
import numpy as np
import sklearn

In [2]:
 df=pd.read_csv("C:\\Users\\emanf\\Desktop\\anime recomendation system\\datasets\\processed\\processed_data.csv")

In [3]:
df.head(30000)

,anime_id,title,score,rank,type,image_url,start_date,end_date,episodes,tags
0,28977,Gintama°,9.05,8,TV,https://cdn.myanimelist.net/images/anime/3/720...,2015-01-01,2016-01-01,51.0,"Action Gintoki, Shinpachi, and Kagura return a..."
1,28977,Gintama°,9.05,8,TV,https://cdn.myanimelist.net/images/anime/3/720...,2015-01-01,2016-01-01,51.0,"Comedy Gintoki, Shinpachi, and Kagura return a..."
2,28977,Gintama°,9.05,8,TV,https://cdn.myanimelist.net/images/anime/3/720...,2015-01-01,2016-01-01,51.0,"Sci-Fi Gintoki, Shinpachi, and Kagura return a..."
3,28977,Gintama°,9.05,8,TV,https://cdn.myanimelist.net/images/anime/3/720...,2015-01-01,2016-01-01,51.0,"Gag Humor Gintoki, Shinpachi, and Kagura retur..."
4,28977,Gintama°,9.05,8,TV,https://cdn.myanimelist.net/images/anime/3/720...,2015-01-01,2016-01-01,51.0,"Historical Gintoki, Shinpachi, and Kagura retu..."
...,...,...,...,...,...,...,...,...,...,...
29995,1056,Good Morning Call,6.06,9933,OVA,https://cdn.myanimelist.net/images/anime/1387/...,2001-01-01,2001-01-01,1.0,Comedy One year has passed since Nao and Uehar...
29996,1056,Good Morning Call,6.06,9933,OVA,https://cdn.myanimelist.net/images/anime/1387/...,2001-01-01,2001-01-01,1.0,Drama One year has passed since Nao and Uehara...
29997,1056,Good Morning Call,6.06,9933,OVA,https://cdn.myanimelist.net/images/anime/1387/...,2001-01-01,2001-01-01,1.0,Romance One year has passed since Nao and Ueha...
29998,1056,Good Morning Call,6.06,9933,OVA,https://cdn.myanimelist.net/images/anime/1387/...,2001-01-01,2001-01-01,1.0,Shoujo One year has passed since Nao and Uehar...


In [4]:
df["anime_id"].duplicated().any()

np.True_

In [5]:
df["anime_id"].duplicated().sum()

np.int64(21810)

In [6]:
new_df = df.drop_duplicates(subset="anime_id", keep="first")

In [7]:
new_df.shape

(8335, 10)

In [8]:
new_df.reset_index(drop=True, inplace=True)

In [9]:
print(new_df.index.min())
print(new_df.index.max())
print(len(new_df))

0
8334
8335


In [10]:
new_df.to_csv("C:\\Users\\emanf\\Desktop\\anime recomendation system\\datasets\\processed\\merged.csv", index=False)

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
cv = CountVectorizer(max_features=8335, stop_words='english')

In [13]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [14]:
cv.get_feature_names_out()

array(['000', '009', '01', ..., 'zoro', 'zorori', 'éclair'],
      shape=(8335,), dtype=object)

In [15]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [16]:
def stem (text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [17]:
new_df ['tags'].apply(stem)

0       action gintoki, shinpachi, and kagura return a...
1                         action sequel to chainsaw man .
2       action hunter devot themselv to accomplish haz...
3       adventur dure their decade-long quest to defea...
4       action after a one-year hiatus, shinpachi shim...
                              ...                        
8330    comedi inform mini anim post on the anime' off...
8331    action in a societi of peopl with anim charact...
8332    action sometim in the future, terror in japan ...
8333    comedi a lost form of magic is reviv from the ...
8334    comedi an onlin motion graphic spin-off seri t...
Name: tags, Length: 8335, dtype: str

In [18]:
new_df['tags'][0]

"Action Gintoki, Shinpachi, and Kagura return as the fun-loving but broke members of the Yorozuya team! Living in an alternate-reality Edo, where swords are prohibited and alien overlords have conquered Japan, they try to thrive on doing whatever work they can get their hands on. However, Shinpachi and Kagura still haven't been paid... Does Gin-chan really spend all that cash playing pachinko? Meanwhile, when Gintoki drunkenly staggers home one night, an alien spaceship crashes nearby. A fatally injured crew member emerges from the ship and gives Gintoki a strange, clock-shaped device, warning him that it is incredibly powerful and must be safeguarded. Mistaking it for his alarm clock, Gintoki proceeds to smash the device the next morning and suddenly discovers that the world outside his apartment has come to a standstill. With Kagura and Shinpachi at his side, he sets off to get the device fixed; though, as usual, nothing is ever that simple for the Yorozuya team. Filled with tongue-i

In [19]:
new_df['tags'][13]

"Drama Tomoya Okazaki and Nagisa Furukawa have graduated from high school, and together, they experience the emotional rollercoaster of growing up. Unable to decide on a course for his future, Tomoya learns the value of a strong work ethic and discovers the strength of Nagisa's support. Through the couple's dedication and unity of purpose, they push forward to confront their personal problems, deepen their old relationships, and create new bonds. Time also moves on in the Illusionary World. As the plains grow cold with the approach of winter, the Illusionary Girl and the Garbage Doll are presented with a difficult situation that reveals the World's true purpose. [Written by MAL Rewrite]"

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
similarity = cosine_similarity(vectors)

In [22]:
print(df.shape)
print(similarity.shape)

(30145, 10)
(8335, 8335)


In [23]:
def recommend(anime):
    
    anime_index = new_df.index[new_df["title"] == anime][0]

    distances = similarity[anime_index]

    anime_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    recommendations = []

    for i in anime_list:
        recommendations.append(new_df.iloc[i[0]]["title"])

    return recommendations



In [24]:
recommend("Good Morning Call")

['Moetan',
 'Oniichan no Koto nanka Zenzen Suki ja Nai n da kara ne!!',
 'Oniichan no Koto nanka Zenzen Suki ja Nai n da kara ne!! Special',
 'Jewelpet Movie: Sweets Dance Princess',
 'Fuujin Monogatari']

In [25]:
import pickle

with open("similarity.pkl", "wb") as file:
    pickle.dump(similarity, file)

In [26]:
print("Number of anime:", len(new_df))

Number of anime: 8335


In [27]:
print("Similarity type:", type(similarity))
print("Similarity shape:", similarity.shape)

Similarity type: <class 'numpy.ndarray'>
Similarity shape: (8335, 8335)


In [28]:
print(
    "Similarity size:",
    similarity.nbytes / (1024 ** 2),
    "MB"
)

Similarity size: 530.0310134887695 MB
